In [1]:
import numpy as np  # noqa: I001
import matplotlib.pyplot as plt
from functools import partial
import jax
import jax.numpy as jnp

# Data Generation and consitilation

In [2]:
@partial(jax.jit, static_argnames=("size", "num_bits"))
def generate_data(key: jax.Array, size: int, num_bits: int = 2) -> jax.Array:
    return jax.random.randint(key, shape=(size,), minval=0, maxval=1 << num_bits)

@partial(jax.jit, static_argnames=("num_bits",))
def get_qam_constellation(num_bits: int = 4) -> jax.Array:
    levels_per_axis = 1 << (num_bits // 2)
    axis = 2 * jnp.arange(levels_per_axis, dtype=jnp.float32) - (levels_per_axis - 1)
    constellation = (axis[:, None] + 1j * axis[None, :]).ravel()
    avg_power = jnp.mean(jnp.abs(constellation) ** 2)
    return constellation / jnp.sqrt(avg_power)

@jax.jit
def get_qpsk_constellation() -> jax.Array:
    angles = jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)
    return jnp.exp(1j * angles)

@partial(jax.jit, static_argnames=("num_bits",))
def get_psk_constellation(num_bits: int = 2) -> jax.Array:
    m = 1 << num_bits
    angles = jnp.pi / m + 2.0 * jnp.pi * jnp.arange(m) / m
    return jnp.exp(1j * angles)

@partial(jax.jit, static_argnames=("num_bits",))
def get_apsk_constellation(num_bits: int = 4, gamma: float = 2.75) -> jax.Array:
    if num_bits == 2:
        angles = jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)
        pts = jnp.exp(1j * angles)
    elif num_bits == 3:
        r1 = jnp.exp(1j * (jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)))
        r2 = gamma * jnp.exp(1j * (jnp.arange(4) * (jnp.pi / 2.0)))
        pts = jnp.concatenate([r1, r2])
    elif num_bits == 5:
        r1 = jnp.exp(1j * (jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)))
        r2 = gamma * jnp.exp(1j * (jnp.pi / 12.0 + jnp.arange(12) * (jnp.pi / 6.0)))
        r3 = (gamma * 1.8) * jnp.exp(1j * (jnp.arange(16) * (jnp.pi / 8.0)))
        pts = jnp.concatenate([r1, r2, r3])
    elif num_bits == 6:
        r1 = jnp.exp(1j * (jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)))
        r2 = gamma * jnp.exp(1j * (jnp.pi / 12.0 + jnp.arange(12) * (jnp.pi / 6.0)))
        r3 = (gamma * 1.7) * jnp.exp(1j * (jnp.pi / 20.0 + jnp.arange(20) * (jnp.pi / 10.0)))
        r4 = (gamma * 2.4) * jnp.exp(1j * (jnp.arange(28) * (2.0 * jnp.pi / 28.0)))
        pts = jnp.concatenate([r1, r2, r3, r4])
    else:
        r1 = jnp.exp(1j * (jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)))
        r2 = gamma * jnp.exp(1j * (jnp.pi / 12.0 + jnp.arange(12) * (jnp.pi / 6.0)))
        pts = jnp.concatenate([r1, r2])
    return pts / jnp.sqrt(jnp.mean(jnp.abs(pts) ** 2))

@jax.jit
def encode_symbols(data: jax.Array, constellation: jax.Array) -> jax.Array:
    return constellation[data]

@jax.jit
def demap_symbols(received_symbols: jax.Array, constellation: jax.Array) -> jax.Array:
    c_power = 0.5 * (constellation.real**2 + constellation.imag**2)
    scores = (
        received_symbols.real[..., None] * constellation.real
        + received_symbols.imag[..., None] * constellation.imag
        - c_power
    )
    return jnp.argmax(scores, axis=-1)


# Noise and jitter

In [3]:
def add_awgn(symbols: jax.Array, snr_db: float, key: jax.Array) -> jax.Array:
    """Adds circularly symmetric complex Gaussian noise at target Es/N0 (dB)."""
    snr_linear = 10.0 ** (snr_db / 10.0)
    signal_power = jnp.mean(jnp.abs(symbols) ** 2)
    noise_power = signal_power / snr_linear
    noise_per_axis = jnp.sqrt(noise_power / 2.0)

    key_r, key_i = jax.random.split(key)
    noise = noise_per_axis * (
        jax.random.normal(key_r, shape=symbols.shape)
        + 1j * jax.random.normal(key_i, shape=symbols.shape)
    )
    return symbols + noise

def add_cfo(symbols: jax.Array, cfo_norm: float) -> jax.Array:
    theta = 2.0 * jnp.pi * cfo_norm * jnp.arange(symbols.shape[0])
    phasor = jnp.exp(1j * theta)
    return symbols * phasor

def add_phase_noise(symbols: jax.Array, phase_noise_std: float, key: jax.Array) -> jax.Array:
    noise = jax.random.normal(key, shape=symbols.shape) * phase_noise_std
    cum_noise = jnp.cumsum(noise)
    phasor = jnp.exp(1j * cum_noise)
    return symbols * phasor

def add_iq_imbalance(symbols: jax.Array, amp_imb: float, phase_imb_rad: float) -> jax.Array:
    I = symbols.real
    Q = symbols.imag
    Q = amp_imb * (Q * jnp.cos(phase_imb_rad) + I * jnp.sin(phase_imb_rad))
    return I + 1j * Q

def add_hpa_rapp(symbols: jax.Array, v_sat: float = 0.9, p: float = 2.0) -> jax.Array:
    gain_compression = (1.0 + (jnp.abs(symbols) / v_sat) ** (2.0 * p)) ** (1.0 / (2.0 * p))
    return symbols / gain_compression

def add_dc_offset(symbols: jax.Array, dc_i: float, dc_q: float) -> jax.Array:
    return symbols + (dc_i + 1j * dc_q)

def get_multipath_taps(
    key: jax.Array,
    num_taps: int = 3,
    decay: float = 0.5,
    k_factor: float = 0.0
) -> jax.Array:
    """Generates normalized Rician/Rayleigh multi-tap channel impulse response.
    
    If k_factor == 0.0, pure Rayleigh fading (NLOS).
    If k_factor > 0.0, Rician fading with dominant LOS component on tap 0.
    """
    delays = jnp.arange(num_taps)
    pdp = jnp.exp(-decay * delays)
    pdp = pdp / jnp.sum(pdp)

    key_r, key_i = jax.random.split(key)
    diffuse = (
        jax.random.normal(key_r, (num_taps,))
        + 1j * jax.random.normal(key_i, (num_taps,))
    ) / jnp.sqrt(2.0)

    los = jnp.zeros(num_taps, dtype=jnp.complex64).at[0].set(1.0 + 0.0j)
    taps = (
        jnp.sqrt(k_factor / (k_factor + 1.0)) * los
        + jnp.sqrt(1.0 / (k_factor + 1.0)) * diffuse
    )
    taps = taps * jnp.sqrt(pdp)
    return taps / jnp.sqrt(jnp.sum(jnp.abs(taps) ** 2))

def add_multipath_fading(symbols: jax.Array, taps: jax.Array) -> jax.Array:
    """Applies multi-tap channel filtering via FIR convolution (mode='same')."""
    return jnp.convolve(symbols, taps, mode="same")

def quantize_uniform(signal: jax.Array, num_bits: int = 8, v_max: float = 1.5) -> jax.Array:
    """Uniform mid-tread quantization on I and Q axes independently."""
    num_levels = 2 ** num_bits
    step = (2.0 * v_max) / (num_levels - 1)
    real_q = jnp.clip(jnp.round(signal.real / step) * step, -v_max, v_max)
    imag_q = jnp.clip(jnp.round(signal.imag / step) * step, -v_max, v_max)
    return real_q + 1j * imag_q

def add_dac_quantization(symbols: jax.Array, num_bits: int = 10, v_max: float = 1.5) -> jax.Array:
    """TX Digital-to-Analog Converter (DAC) finite bit-depth quantization."""
    return quantize_uniform(symbols, num_bits=num_bits, v_max=v_max)

def add_adc_quantization(symbols: jax.Array, num_bits: int = 8, v_max: float = 1.5) -> jax.Array:
    """RX Analog-to-Digital Converter (ADC) finite bit-depth quantization."""
    return quantize_uniform(symbols, num_bits=num_bits, v_max=v_max)


In [4]:
@partial(jax.jit, static_argnames=("dac_bits", "adc_bits"))
def noise_pipeline(
    symbols: jax.Array,
    rng: jax.Array,
    dac_bits: int = 10,
    v_dac_max: float = 1.5,
    amp_imb: float = 1.0,
    phase_imb_rad: float = 0.1,
    v_sat: float = 13.0,
    p: float = 2.0,
    taps: jax.Array = jnp.arange(2),
    cfo_norm: float = 2.0,
    phase_noise_std: float = 1.0,
    snr_db: float = 2.0,
    adc_bits: int = 10,
    v_adc_max: float = 1.5,
) -> jax.Array:
    symbols = add_dac_quantization(symbols, num_bits=dac_bits, v_max=v_dac_max)
    symbols = add_iq_imbalance(symbols, amp_imb=amp_imb, phase_imb_rad=phase_imb_rad)
    symbols = add_hpa_rapp(symbols, v_sat=v_sat, p=p)
    symbols = add_multipath_fading(symbols, taps=taps)
    symbols = add_cfo(symbols, cfo_norm=cfo_norm)
    symbols = add_phase_noise(symbols, phase_noise_std=phase_noise_std, key=rng)
    symbols = add_awgn(symbols, snr_db=snr_db, key=rng)
    symbols = add_adc_quantization(symbols, num_bits=adc_bits, v_max=v_adc_max)
    return symbols


In [5]:
def invert_dc_offset(symbols:jax.Array,dc_i:float,dc_q:float)->jax.Array:
    return symbols-(dc_i+1j*dc_q)

def invert_cfo(symbols:jax.Array,cfo_norm)->jax.Array:
    theta=2*jnp.pi*cfo_norm*jnp.arange(symbols.shape[0])
    return symbols*jnp.exp(-1j*theta)

def invert_iq_imbalance(symbols:jax.Array,amp_imb:float,phase_imb_rad:float)->jax.Array:
    I=symbols.real
    Q=symbols.imag
    Q_clean = (Q / amp_imb - I * jnp.sin(phase_imb_rad)) /    jnp.cos(phase_imb_rad)
    return I+1j*Q_clean

def invert_hpa_rapp(symbols:jax.Array,v_sat:float=0.9,p:float=2.0)->jax.Array:
    ratio=jnp.minimum(jnp.abs(symbols)/v_sat,0.9999)
    gain_expansion=(1.0-ratio**(2.0*p))**(1.0/(2.0*p))
    return symbols/gain_expansion

def invert_multipath_fading(symbols:jax.Array,taps:jax.Array,reg:float=1e-4)->jax.Array:
    n=symbols.shape[0]
    delay=(taps.shape[0]-1)//2
    shift=jnp.exp(1j*2.0*jnp.pi*delay*jnp.arange(n)/n)
    H=jnp.fft.fft(taps,n=n)*shift
    X_hat=(jnp.fft.fft(symbols)*jnp.conj(H))/(jnp.abs(H)**2+reg)
    return jnp.fft.ifft(X_hat)


In [6]:
@jax.jit
def deterministic_denoise_pipeline(
    symbols:jax.Array,
    cfo_norm:float=0.0,
    taps:jax.Array=jnp.array([1.0],dtype=jnp.complex64),
    v_sat:float=13.0,
    p:float=2.0,
    dc_i:float=0.0,
    dc_q:float=0.0,
    amp_imb:float=1.0,
    phase_imb_rad:float=0.0,
)->jax.Array:
    symbols=invert_cfo(symbols,cfo_norm=cfo_norm)
    symbols=invert_multipath_fading(symbols,taps=taps)
    symbols=invert_hpa_rapp(symbols,v_sat=v_sat,p=p)
    symbols=invert_dc_offset(symbols,dc_i=dc_i,dc_q=dc_q)
    symbols=invert_iq_imbalance(symbols,amp_imb=amp_imb,phase_imb_rad=phase_imb_rad)
    return symbols


## Interactive Transceiver & Denoising Dashboard

This dashboard visualizes the **Forward Impairment Pipeline** (left) side-by-side with the **Stage 1 Deterministic Denoising Pipeline** (right).

* **Auto-Sync Denoiser (Checked):** Automatically feeds the exact channel parameters into  to demonstrate perfect physics inversion.
* **Auto-Sync Denoiser (Unchecked):** Lets you manually tune receiver inversion sliders () to simulate parameter estimation mismatch (like manually dialing a receiver knob).
* **Residual Noise:** Notice how deterministic inversion de-rotates CFO, squares up IQ imbalance, and removes multipath echoes, leaving only the stochastic AWGN cloud and Wiener phase noise for Stage 2 (Flow Matching).


In [ ]:
# --- Interactive Transceiver & Denoising Dashboard ---
import ipywidgets as widgets
from IPython.display import display

key = jax.random.key(42)
key_data, key_noise = jax.random.split(key)

num_bits = 4  # 16-QAM
constellation = get_qam_constellation(num_bits=num_bits)
tx_data = generate_data(key_data, size=1500, num_bits=num_bits)
tx_symbols = encode_symbols(tx_data, constellation)

@widgets.interact(
    # --- 1. Forward Channel Impairments ---
    dac_bits=widgets.IntSlider(value=10, min=2, max=12, step=1, description="1. DAC Bits:", continuous_update=True),
    amp_imb=widgets.FloatSlider(value=1.0, min=0.7, max=1.3, step=0.01, description="2. Amp Imb:", continuous_update=True),
    phase_imb_rad=widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.01, description="3. Phase Imb:", continuous_update=True),
    v_sat=widgets.FloatSlider(value=2.0, min=0.5, max=3.0, step=0.05, description="4. HPA Vsat:", continuous_update=True),
    p=widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description="5. HPA Knee:", continuous_update=True),
    tap2_echo=widgets.FloatSlider(value=0.0, min=0.0, max=0.8, step=0.02, description="6. Echo Gain:", continuous_update=True),
    cfo_norm=widgets.FloatSlider(value=0.0, min=-0.005, max=0.005, step=0.0001, description="7. CFO Norm:", readout_format=".4f", continuous_update=True),
    phase_noise_std=widgets.FloatSlider(value=0.0, min=0.0, max=0.1, step=0.002, description="8. Phase Noise:", readout_format=".3f", continuous_update=True),
    snr_db=widgets.FloatSlider(value=30.0, min=-5.0, max=45.0, step=0.5, description="9. SNR (dB):", continuous_update=True),
    adc_bits=widgets.IntSlider(value=10, min=2, max=12, step=1, description="10. ADC Bits:", continuous_update=True),
    # --- 2. Deterministic Denoiser Controls ---
    enable_denoiser=widgets.Checkbox(value=True, description="Enable Denoiser"),
    auto_sync_denoiser=widgets.Checkbox(value=True, description="Auto-Sync Denoiser to Channel"),
    denoise_cfo=widgets.FloatSlider(value=0.0, min=-0.005, max=0.005, step=0.0001, description="Inv CFO:", readout_format=".4f", continuous_update=True),
    denoise_echo=widgets.FloatSlider(value=0.0, min=0.0, max=0.8, step=0.02, description="Inv Echo:", continuous_update=True),
    denoise_amp_imb=widgets.FloatSlider(value=1.0, min=0.7, max=1.3, step=0.01, description="Inv Amp Imb:", continuous_update=True),
    denoise_phase_imb=widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.01, description="Inv Phase Imb:", continuous_update=True),
    denoise_vsat=widgets.FloatSlider(value=2.0, min=0.5, max=3.0, step=0.05, description="Inv HPA Vsat:", continuous_update=True),
)
def plot_transceiver_dashboard(
    dac_bits=10,
    amp_imb=1.0,
    phase_imb_rad=0.0,
    v_sat=2.0,
    p=2.0,
    tap2_echo=0.0,
    cfo_norm=0.0,
    phase_noise_std=0.0,
    snr_db=30.0,
    adc_bits=10,
    enable_denoiser=True,
    auto_sync_denoiser=True,
    denoise_cfo=0.0,
    denoise_echo=0.0,
    denoise_amp_imb=1.0,
    denoise_phase_imb=0.0,
    denoise_vsat=2.0,
):
    # Channel multipath filter
    taps_fwd = jnp.array([1.0, tap2_echo], dtype=jnp.complex64)
    taps_fwd = taps_fwd / jnp.sqrt(jnp.sum(jnp.abs(taps_fwd) ** 2))

    # 1. Forward Impairment Transmission
    rx_noisy = noise_pipeline(
        tx_symbols,
        key_noise,
        dac_bits=dac_bits,
        v_dac_max=1.5,
        amp_imb=amp_imb,
        phase_imb_rad=phase_imb_rad,
        v_sat=v_sat,
        p=p,
        taps=taps_fwd,
        cfo_norm=cfo_norm,
        phase_noise_std=phase_noise_std,
        snr_db=snr_db,
        adc_bits=adc_bits,
        v_adc_max=1.5,
    )

    ser_noisy = float(jnp.mean(demap_symbols(rx_noisy, constellation) != tx_data))

    # 2. Stage 1 Deterministic Denoising
    if enable_denoiser:
        if auto_sync_denoiser:
            inv_cfo_val = cfo_norm
            inv_taps = taps_fwd
            inv_amp_val = amp_imb
            inv_phase_val = phase_imb_rad
            inv_vsat_val = v_sat
        else:
            inv_cfo_val = denoise_cfo
            inv_taps = jnp.array([1.0, denoise_echo], dtype=jnp.complex64)
            inv_taps = inv_taps / jnp.sqrt(jnp.sum(jnp.abs(inv_taps) ** 2))
            inv_amp_val = denoise_amp_imb
            inv_phase_val = denoise_phase_imb
            inv_vsat_val = denoise_vsat

        rx_denoised = deterministic_denoise_pipeline(
            rx_noisy,
            cfo_norm=inv_cfo_val,
            taps=inv_taps,
            v_sat=inv_vsat_val,
            p=p,
            dc_i=0.0,
            dc_q=0.0,
            amp_imb=inv_amp_val,
            phase_imb_rad=inv_phase_val,
        )
        ser_denoised = float(jnp.mean(demap_symbols(rx_denoised, constellation) != tx_data))
    else:
        rx_denoised = rx_noisy
        ser_denoised = ser_noisy

    # Side-by-Side Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Left: Received Distorted
    ax1.scatter(np.array(rx_noisy.real), np.array(rx_noisy.imag), s=10, alpha=0.35, c=np.arange(rx_noisy.shape[0]), cmap="viridis")
    ax1.scatter(np.array(constellation.real), np.array(constellation.imag), color="red", marker="x", alpha=0.6, s=60, linewidths=2)
    ax1.axhline(0, color="gray", linestyle="--", alpha=0.3)
    ax1.axvline(0, color="gray", linestyle="--", alpha=0.3)
    ax1.set_xlim(-2.0, 2.0)
    ax1.set_ylim(-2.0, 2.0)
    ax1.set_title(f"Received Noisy RX (SER: {ser_noisy:.2%})")
    ax1.set_xlabel("In-Phase (I)")
    ax1.set_ylabel("Quadrature (Q)")
    ax1.grid(True, alpha=0.3)

    # Right: Denoised
    ax2.scatter(np.array(rx_denoised.real), np.array(rx_denoised.imag), s=10, alpha=0.35, c=np.arange(rx_denoised.shape[0]), cmap="plasma")
    ax2.scatter(np.array(constellation.real), np.array(constellation.imag), color="red", marker="x", alpha=0.6, s=60, linewidths=2)
    ax2.axhline(0, color="gray", linestyle="--", alpha=0.3)
    ax2.axvline(0, color="gray", linestyle="--", alpha=0.3)
    ax2.set_xlim(-2.0, 2.0)
    ax2.set_ylim(-2.0, 2.0)
    ax2.set_title(f"Stage 1 Denoised (SER: {ser_denoised:.2%})")
    ax2.set_xlabel("In-Phase (I)")
    ax2.set_ylabel("Quadrature (Q)")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


interactive(children=(IntSlider(value=10, description='1. DAC Bits:', max=12, min=2), FloatSlider(value=1.0, d…

In [8]:
# --- Interactive APSK Transceiver & Denoising Dashboard ---
import ipywidgets as widgets
from IPython.display import display

@partial(jax.jit, static_argnames=("num_bits",))
def get_apsk_constellation(num_bits: int = 4, gamma: float = 2.75) -> jax.Array:
    if num_bits == 2:
        angles = jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)
        pts = jnp.exp(1j * angles)
    elif num_bits == 3:
        r1 = jnp.exp(1j * (jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)))
        r2 = gamma * jnp.exp(1j * (jnp.arange(4) * (jnp.pi / 2.0)))
        pts = jnp.concatenate([r1, r2])
    elif num_bits == 5:
        r1 = jnp.exp(1j * (jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)))
        r2 = gamma * jnp.exp(1j * (jnp.pi / 12.0 + jnp.arange(12) * (jnp.pi / 6.0)))
        r3 = (gamma * 1.8) * jnp.exp(1j * (jnp.arange(16) * (jnp.pi / 8.0)))
        pts = jnp.concatenate([r1, r2, r3])
    elif num_bits == 6:
        r1 = jnp.exp(1j * (jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)))
        r2 = gamma * jnp.exp(1j * (jnp.pi / 12.0 + jnp.arange(12) * (jnp.pi / 6.0)))
        r3 = (gamma * 1.7) * jnp.exp(1j * (jnp.pi / 20.0 + jnp.arange(20) * (jnp.pi / 10.0)))
        r4 = (gamma * 2.4) * jnp.exp(1j * (jnp.arange(28) * (2.0 * jnp.pi / 28.0)))
        pts = jnp.concatenate([r1, r2, r3, r4])
    else:
        r1 = jnp.exp(1j * (jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)))
        r2 = gamma * jnp.exp(1j * (jnp.pi / 12.0 + jnp.arange(12) * (jnp.pi / 6.0)))
        pts = jnp.concatenate([r1, r2])
    return pts / jnp.sqrt(jnp.mean(jnp.abs(pts) ** 2))

key_apsk = jax.random.key(99)
key_data_apsk, key_noise_apsk = jax.random.split(key_apsk)

@widgets.interact(
    num_bits=widgets.Dropdown(options=[2, 3, 4, 5, 6], value=4, description="APSK Bits:"),
    gamma=widgets.FloatSlider(value=2.75, min=1.5, max=4.0, step=0.05, description="Ring Ratio γ:", continuous_update=True),
    dac_bits=widgets.IntSlider(value=10, min=2, max=12, step=1, description="1. DAC Bits:", continuous_update=True),
    amp_imb=widgets.FloatSlider(value=1.0, min=0.7, max=1.3, step=0.01, description="2. Amp Imb:", continuous_update=True),
    phase_imb_rad=widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.01, description="3. Phase Imb:", continuous_update=True),
    v_sat=widgets.FloatSlider(value=2.0, min=0.5, max=3.0, step=0.05, description="4. HPA Vsat:", continuous_update=True),
    p=widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description="5. HPA Knee:", continuous_update=True),
    tap2_echo=widgets.FloatSlider(value=0.0, min=0.0, max=0.8, step=0.02, description="6. Echo Gain:", continuous_update=True),
    cfo_norm=widgets.FloatSlider(value=0.0, min=-0.005, max=0.005, step=0.0001, description="7. CFO Norm:", readout_format=".4f", continuous_update=True),
    phase_noise_std=widgets.FloatSlider(value=0.0, min=0.0, max=0.1, step=0.002, description="8. Phase Noise:", readout_format=".3f", continuous_update=True),
    snr_db=widgets.FloatSlider(value=30.0, min=-5.0, max=45.0, step=0.5, description="9. SNR (dB):", continuous_update=True),
    adc_bits=widgets.IntSlider(value=10, min=2, max=12, step=1, description="10. ADC Bits:", continuous_update=True),
    enable_denoiser=widgets.Checkbox(value=True, description="Enable Denoiser"),
    auto_sync_denoiser=widgets.Checkbox(value=True, description="Auto-Sync Denoiser to Channel"),
    denoise_cfo=widgets.FloatSlider(value=0.0, min=-0.005, max=0.005, step=0.0001, description="Inv CFO:", readout_format=".4f", continuous_update=True),
    denoise_echo=widgets.FloatSlider(value=0.0, min=0.0, max=0.8, step=0.02, description="Inv Echo:", continuous_update=True),
    denoise_amp_imb=widgets.FloatSlider(value=1.0, min=0.7, max=1.3, step=0.01, description="Inv Amp Imb:", continuous_update=True),
    denoise_phase_imb=widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.01, description="Inv Phase Imb:", continuous_update=True),
    denoise_vsat=widgets.FloatSlider(value=2.0, min=0.5, max=3.0, step=0.05, description="Inv HPA Vsat:", continuous_update=True),
)
def plot_apsk_dashboard(
    num_bits=4,
    gamma=2.75,
    dac_bits=10,
    amp_imb=1.0,
    phase_imb_rad=0.0,
    v_sat=2.0,
    p=2.0,
    tap2_echo=0.0,
    cfo_norm=0.0,
    phase_noise_std=0.0,
    snr_db=30.0,
    adc_bits=10,
    enable_denoiser=True,
    auto_sync_denoiser=True,
    denoise_cfo=0.0,
    denoise_echo=0.0,
    denoise_amp_imb=1.0,
    denoise_phase_imb=0.0,
    denoise_vsat=2.0,
):
    constellation = get_apsk_constellation(num_bits=num_bits, gamma=gamma)
    tx_data = generate_data(key_data_apsk, size=1500, num_bits=num_bits)
    tx_symbols = encode_symbols(tx_data, constellation)

    taps_fwd = jnp.array([1.0, tap2_echo], dtype=jnp.complex64)
    taps_fwd = taps_fwd / jnp.sqrt(jnp.sum(jnp.abs(taps_fwd) ** 2))

    rx_noisy = noise_pipeline(
        tx_symbols,
        key_noise_apsk,
        dac_bits=dac_bits,
        v_dac_max=1.5,
        amp_imb=amp_imb,
        phase_imb_rad=phase_imb_rad,
        v_sat=v_sat,
        p=p,
        taps=taps_fwd,
        cfo_norm=cfo_norm,
        phase_noise_std=phase_noise_std,
        snr_db=snr_db,
        adc_bits=adc_bits,
        v_adc_max=1.5,
    )

    ser_noisy = float(jnp.mean(demap_symbols(rx_noisy, constellation) != tx_data))

    if enable_denoiser:
        if auto_sync_denoiser:
            inv_cfo_val = cfo_norm
            inv_taps = taps_fwd
            inv_amp_val = amp_imb
            inv_phase_val = phase_imb_rad
            inv_vsat_val = v_sat
        else:
            inv_cfo_val = denoise_cfo
            inv_taps = jnp.array([1.0, denoise_echo], dtype=jnp.complex64)
            inv_taps = inv_taps / jnp.sqrt(jnp.sum(jnp.abs(inv_taps) ** 2))
            inv_amp_val = denoise_amp_imb
            inv_phase_val = denoise_phase_imb
            inv_vsat_val = denoise_vsat

        rx_denoised = deterministic_denoise_pipeline(
            rx_noisy,
            cfo_norm=inv_cfo_val,
            taps=inv_taps,
            v_sat=inv_vsat_val,
            p=p,
            dc_i=0.0,
            dc_q=0.0,
            amp_imb=inv_amp_val,
            phase_imb_rad=inv_phase_val,
        )
        ser_denoised = float(jnp.mean(demap_symbols(rx_denoised, constellation) != tx_data))
    else:
        rx_denoised = rx_noisy
        ser_denoised = ser_noisy

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    ax1.scatter(np.array(rx_noisy.real), np.array(rx_noisy.imag), s=10, alpha=0.35, c=np.arange(rx_noisy.shape[0]), cmap="viridis")
    ax1.scatter(np.array(constellation.real), np.array(constellation.imag), color="red", marker="x", alpha=0.6, s=60, linewidths=2)
    ax1.axhline(0, color="gray", linestyle="--", alpha=0.3)
    ax1.axvline(0, color="gray", linestyle="--", alpha=0.3)
    ax1.set_xlim(-2.0, 2.0)
    ax1.set_ylim(-2.0, 2.0)
    ax1.set_title(f"Received Noisy {1 << num_bits}-APSK (SER: {ser_noisy:.2%})")
    ax1.set_xlabel("In-Phase (I)")
    ax1.set_ylabel("Quadrature (Q)")
    ax1.grid(True, alpha=0.3)

    ax2.scatter(np.array(rx_denoised.real), np.array(rx_denoised.imag), s=10, alpha=0.35, c=np.arange(rx_denoised.shape[0]), cmap="plasma")
    ax2.scatter(np.array(constellation.real), np.array(constellation.imag), color="red", marker="x", alpha=0.6, s=60, linewidths=2)
    ax2.axhline(0, color="gray", linestyle="--", alpha=0.3)
    ax2.axvline(0, color="gray", linestyle="--", alpha=0.3)
    ax2.set_xlim(-2.0, 2.0)
    ax2.set_ylim(-2.0, 2.0)
    ax2.set_title(f"Stage 1 Denoised {1 << num_bits}-APSK (SER: {ser_denoised:.2%})")
    ax2.set_xlabel("In-Phase (I)")
    ax2.set_ylabel("Quadrature (Q)")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


interactive(children=(Dropdown(description='APSK Bits:', index=2, options=(2, 3, 4, 5, 6), value=4), FloatSlid…

In [9]:
import ipywidgets as widgets
from IPython.display import display

@jax.jit
def get_qpsk_constellation() -> jax.Array:
    angles = jnp.pi / 4.0 + jnp.arange(4) * (jnp.pi / 2.0)
    return jnp.exp(1j * angles)

key_qpsk = jax.random.key(77)
key_data_qpsk, key_noise_qpsk = jax.random.split(key_qpsk)

@widgets.interact(
    dac_bits=widgets.IntSlider(value=10, min=2, max=12, step=1, description="1. DAC Bits:", continuous_update=True),
    amp_imb=widgets.FloatSlider(value=1.0, min=0.7, max=1.3, step=0.01, description="2. Amp Imb:", continuous_update=True),
    phase_imb_rad=widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.01, description="3. Phase Imb:", continuous_update=True),
    v_sat=widgets.FloatSlider(value=2.0, min=0.5, max=3.0, step=0.05, description="4. HPA Vsat:", continuous_update=True),
    p=widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description="5. HPA Knee:", continuous_update=True),
    tap2_echo=widgets.FloatSlider(value=0.0, min=0.0, max=0.8, step=0.02, description="6. Echo Gain:", continuous_update=True),
    cfo_norm=widgets.FloatSlider(value=0.0, min=-0.005, max=0.005, step=0.0001, description="7. CFO Norm:", readout_format=".4f", continuous_update=True),
    phase_noise_std=widgets.FloatSlider(value=0.0, min=0.0, max=0.1, step=0.002, description="8. Phase Noise:", readout_format=".3f", continuous_update=True),
    snr_db=widgets.FloatSlider(value=30.0, min=-5.0, max=45.0, step=0.5, description="9. SNR (dB):", continuous_update=True),
    adc_bits=widgets.IntSlider(value=10, min=2, max=12, step=1, description="10. ADC Bits:", continuous_update=True),
    enable_denoiser=widgets.Checkbox(value=True, description="Enable Denoiser"),
    auto_sync_denoiser=widgets.Checkbox(value=True, description="Auto-Sync Denoiser to Channel"),
    denoise_cfo=widgets.FloatSlider(value=0.0, min=-0.005, max=0.005, step=0.0001, description="Inv CFO:", readout_format=".4f", continuous_update=True),
    denoise_echo=widgets.FloatSlider(value=0.0, min=0.0, max=0.8, step=0.02, description="Inv Echo:", continuous_update=True),
    denoise_amp_imb=widgets.FloatSlider(value=1.0, min=0.7, max=1.3, step=0.01, description="Inv Amp Imb:", continuous_update=True),
    denoise_phase_imb=widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.01, description="Inv Phase Imb:", continuous_update=True),
    denoise_vsat=widgets.FloatSlider(value=2.0, min=0.5, max=3.0, step=0.05, description="Inv HPA Vsat:", continuous_update=True),
)
def plot_qpsk_dashboard(
    dac_bits=10,
    amp_imb=1.0,
    phase_imb_rad=0.0,
    v_sat=2.0,
    p=2.0,
    tap2_echo=0.0,
    cfo_norm=0.0,
    phase_noise_std=0.0,
    snr_db=30.0,
    adc_bits=10,
    enable_denoiser=True,
    auto_sync_denoiser=True,
    denoise_cfo=0.0,
    denoise_echo=0.0,
    denoise_amp_imb=1.0,
    denoise_phase_imb=0.0,
    denoise_vsat=2.0,
):
    constellation = get_qpsk_constellation()
    tx_data = generate_data(key_data_qpsk, size=1500, num_bits=2)
    tx_symbols = encode_symbols(tx_data, constellation)

    taps_fwd = jnp.array([1.0, tap2_echo], dtype=jnp.complex64)
    taps_fwd = taps_fwd / jnp.sqrt(jnp.sum(jnp.abs(taps_fwd) ** 2))

    rx_noisy = noise_pipeline(
        tx_symbols,
        key_noise_qpsk,
        dac_bits=dac_bits,
        v_dac_max=1.5,
        amp_imb=amp_imb,
        phase_imb_rad=phase_imb_rad,
        v_sat=v_sat,
        p=p,
        taps=taps_fwd,
        cfo_norm=cfo_norm,
        phase_noise_std=phase_noise_std,
        snr_db=snr_db,
        adc_bits=adc_bits,
        v_adc_max=1.5,
    )

    ser_noisy = float(jnp.mean(demap_symbols(rx_noisy, constellation) != tx_data))

    if enable_denoiser:
        if auto_sync_denoiser:
            inv_cfo_val = cfo_norm
            inv_taps = taps_fwd
            inv_amp_val = amp_imb
            inv_phase_val = phase_imb_rad
            inv_vsat_val = v_sat
        else:
            inv_cfo_val = denoise_cfo
            inv_taps = jnp.array([1.0, denoise_echo], dtype=jnp.complex64)
            inv_taps = inv_taps / jnp.sqrt(jnp.sum(jnp.abs(inv_taps) ** 2))
            inv_amp_val = denoise_amp_imb
            inv_phase_val = denoise_phase_imb
            inv_vsat_val = denoise_vsat

        rx_denoised = deterministic_denoise_pipeline(
            rx_noisy,
            cfo_norm=inv_cfo_val,
            taps=inv_taps,
            v_sat=inv_vsat_val,
            p=p,
            dc_i=0.0,
            dc_q=0.0,
            amp_imb=inv_amp_val,
            phase_imb_rad=inv_phase_val,
        )
        ser_denoised = float(jnp.mean(demap_symbols(rx_denoised, constellation) != tx_data))
    else:
        rx_denoised = rx_noisy
        ser_denoised = ser_noisy

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    ax1.scatter(np.array(rx_noisy.real), np.array(rx_noisy.imag), s=10, alpha=0.35, c=np.arange(rx_noisy.shape[0]), cmap="viridis")
    ax1.scatter(np.array(constellation.real), np.array(constellation.imag), color="red", marker="x", alpha=0.6, s=80, linewidths=2)
    ax1.axhline(0, color="gray", linestyle="--", alpha=0.3)
    ax1.axvline(0, color="gray", linestyle="--", alpha=0.3)
    ax1.set_xlim(-2.0, 2.0)
    ax1.set_ylim(-2.0, 2.0)
    ax1.set_title(f"Received Noisy QPSK (SER: {ser_noisy:.2%})")
    ax1.set_xlabel("In-Phase (I)")
    ax1.set_ylabel("Quadrature (Q)")
    ax1.grid(True, alpha=0.3)

    ax2.scatter(np.array(rx_denoised.real), np.array(rx_denoised.imag), s=10, alpha=0.35, c=np.arange(rx_denoised.shape[0]), cmap="plasma")
    ax2.scatter(np.array(constellation.real), np.array(constellation.imag), color="red", marker="x", alpha=0.6, s=80, linewidths=2)
    ax2.axhline(0, color="gray", linestyle="--", alpha=0.3)
    ax2.axvline(0, color="gray", linestyle="--", alpha=0.3)
    ax2.set_xlim(-2.0, 2.0)
    ax2.set_ylim(-2.0, 2.0)
    ax2.set_title(f"Stage 1 Denoised QPSK (SER: {ser_denoised:.2%})")
    ax2.set_xlabel("In-Phase (I)")
    ax2.set_ylabel("Quadrature (Q)")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


interactive(children=(IntSlider(value=10, description='1. DAC Bits:', max=12, min=2), FloatSlider(value=1.0, d…